# ST Score Restore — Stage 11 V2d Colab GPU Detector Benchmark

**Amaç:** Frozen V2a restore modelini değiştirmeden, `%100` teacher-ground-truth karşısında inference-only semantic detector adayını hızlıca benchmark etmek.

Bu notebook:
- sadece `development` corpus kullanır,
- held-out veriye dokunmaz,
- training / fine-tuning / optimizer çalıştırmaz,
- detector çıktısını ground-truth yapmaz,
- GPU sonucunu production/Stage 12 kanıtı saymaz,
- seçilen detector için son canonical CPU rerun gerektirir.

İlk aday: `BreezeWhite/oemer` ONNX segmentation, commit `dbe2a933...` (MIT repository license; checkpoint lisansı production için ayrıca review_required).

In [ ]:
# 1) Runtime + dependencies
!nvidia-smi || true
!apt-get -qq update
!apt-get -qq install -y poppler-utils
!pip -q install "git+https://github.com/BreezeWhite/oemer@dbe2a933d630d0f74805d717960eb259473f5978"

import os, sys, json, hashlib, subprocess, shutil, time
from pathlib import Path
import numpy as np
import cv2
import torch

print("python", sys.version)
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime gerekli: Colab > Runtime > Change runtime type > GPU")

In [ ]:
# 2) Mount Drive and locate repository
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_EVAL')
if not DRIVE_ROOT.exists():
    raise FileNotFoundError(f"Drive folder bulunamadı: {DRIVE_ROOT}")

REPO = Path('/content/st-score-restore-engine')
if REPO.exists():
    shutil.rmtree(REPO)

!git clone -q -b stage11-v2c-semantic-detector-corpus-expansion https://github.com/khfy7wpr5p-maker/st-score-restore-engine.git /content/st-score-restore-engine
!pip -q install -e /content/st-score-restore-engine

print("repo ready", REPO)

In [ ]:
# 3) Exact-byte target inventory (size filter first, then SHA-256)
TARGETS = {
    "restore_model": {
        "sha256": "7ff4023466f6b18eda41d9e8af7a9f6858429354621781dd9bd93b26210ba234",
        "size": 7817857,
        "kind": "torchscript",
    },
    "beethoven": {
        "sha256": "c25a5c5979ae076f8fc3607926ddb1d6aeb96a394498c2c1ebc54c27d884053c",
        "size": 1182561,
        "kind": "pdf",
        "pages": [1,2,3,4],
        "page_ids": ["beethoven-op48-no3-p1","beethoven-op48-no3-p2","beethoven-op48-no3-p3","beethoven-op48-no3-p4"],
    },
    "wikimedia": {
        "sha256": "36484c2bfbb57643d992ca77fc0c8f9de0991f52d035d91bb0c780f097de3dcb",
        "size": 34636,
        "kind": "png",
        "pages": [1],
        "page_ids": ["wikimedia-guitar-technical-exercise-no1-p1"],
    },
    "barley": {
        "sha256": "6b3044422b4df58dc4e458cba3de75fd99c88e13c2060498db191238cfdbac6e",
        "size": 84689,
        "kind": "pdf",
        "pages": [1,2],
        "page_ids": ["barley-your-face-your-tongue-your-wit-p1","barley-your-face-your-tongue-your-wit-p2"],
    },
    "carulli": {
        "sha256": "db20e9ce755aa56dd9dbb0436a37a48545442e7961e471f813cde7aaa8fc0f22",
        "size": 4662523,
        "kind": "pdf",
        "pages": [2,3,4,5,6,7,8,9,10],
        "page_ids": [f"carulli-morceaux-faciles-p{i}" for i in range(2,11)],
    },
    "bach": {
        "sha256": "692d4317375048b9d520b4d756c3ce992e96ca82b10c3026a5925f1a878fc959",
        "size": 3235494,
        "kind": "pdf",
        "pages": [4,10,24,40],
        "page_ids": ["bach-anna-magdalena-p4","bach-anna-magdalena-p10","bach-anna-magdalena-p24","bach-anna-magdalena-p40"],
    },
}

def sha256_file(path: Path, chunk=1024*1024):
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

size_to_keys = {}
for key, spec in TARGETS.items():
    size_to_keys.setdefault(spec["size"], []).append(key)

found = {}
for p in DRIVE_ROOT.rglob("*"):
    if not p.is_file():
        continue
    try:
        size = p.stat().st_size
    except OSError:
        continue
    if size not in size_to_keys:
        continue
    digest = sha256_file(p)
    for key in size_to_keys[size]:
        if digest == TARGETS[key]["sha256"]:
            found[key] = p

missing = sorted(set(TARGETS) - set(found))
print("found:", {k: str(v) for k,v in found.items()})
if missing:
    raise FileNotFoundError(f"Exact-byte targets missing: {missing}")

In [ ]:
# 4) Materialize the final 100% teacher ground truth (source-only review + explicit resolutions)
from st_score_restore.stage11_v2c_teacher_review import materialize_teacher_review_manifest
from st_score_restore.stage11_v2d_detector_benchmark import (
    benchmark_coverage_from_present_pairs,
    teacher_present_class_pairs,
)

base = json.loads((REPO/'evidence/stage11/v2c/v2c-expected-class-manifest.v1.json').read_text())
overlay = json.loads((REPO/'evidence/stage11/v2c/v2c-teacher-review-overlay.v1.json').read_text())
resolution = json.loads((REPO/'evidence/stage11/v2c/v2c-teacher-review-resolution.v1.json').read_text())
teacher = materialize_teacher_review_manifest(base, overlay, resolution)["manifest"]
eligible_pairs = teacher_present_class_pairs(teacher)

print("teacher present class-page pairs:", len(eligible_pairs))
assert len(eligible_pairs) == 171

In [ ]:
# 5) Render exact selected pages at 72 DPI (pdftoppm) and prepare page map
WORK = Path('/content/v2d_work')
SRC_DIR = WORK/'source'
SRC_DIR.mkdir(parents=True, exist_ok=True)

page_paths = {}

def render_pdf_page(pdf: Path, page_num: int, out_png: Path):
    base = out_png.with_suffix("")
    subprocess.run([
        "pdftoppm", "-f", str(page_num), "-l", str(page_num),
        "-singlefile", "-r", "72", "-png", str(pdf), str(base)
    ], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if not out_png.exists():
        alt = Path(str(base)+".png")
        if alt.exists() and alt != out_png:
            alt.replace(out_png)
    if not out_png.exists():
        raise FileNotFoundError(out_png)

for family, spec in TARGETS.items():
    if family == "restore_model":
        continue
    for page_num, page_id in zip(spec["pages"], spec["page_ids"]):
        out = SRC_DIR/f"{page_id}.png"
        if spec["kind"] == "pdf":
            render_pdf_page(found[family], page_num, out)
        else:
            shutil.copy2(found[family], out)
        page_paths[page_id] = out

print("prepared pages:", len(page_paths))
assert len(page_paths) == 20

In [ ]:
# 6) Generate exploratory restored pages with the exact frozen package.
# GPU is used only to accelerate exploration. These outputs are NOT canonical evidence.
REST_DIR = WORK/'restored_gpu_exploratory'
REST_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda")
model = torch.jit.load(str(found["restore_model"]), map_location=device)
model = model.to(device).eval()

PATCH = 512
OVERLAP = 64
STRIDE = PATCH - OVERLAP

def restore_gray(gray_u8: np.ndarray) -> np.ndarray:
    h, w = gray_u8.shape
    acc = np.zeros((h, w), np.float32)
    cnt = np.zeros((h, w), np.float32)
    with torch.inference_mode():
        for y in range(0, h, STRIDE):
            for x in range(0, w, STRIDE):
                y2, x2 = min(y+PATCH, h), min(x+PATCH, w)
                tile = np.full((PATCH, PATCH), 255, np.uint8)
                tile[:y2-y, :x2-x] = gray_u8[y:y2, x:x2]
                inp = torch.from_numpy(tile.astype(np.float32)/255.0)[None,None].to(device)
                out = model(inp)
                if isinstance(out, (tuple, list)):
                    out = out[0]
                arr = out.detach().float().cpu().numpy().squeeze()
                arr = np.clip(arr, 0.0, 1.0)
                arr = (arr*255.0 + 0.5).astype(np.uint8)
                crop = arr[:y2-y, :x2-x].astype(np.float32)
                acc[y:y2, x:x2] += crop
                cnt[y:y2, x:x2] += 1.0
    return np.clip(acc/np.maximum(cnt,1.0),0,255).astype(np.uint8)

restored_paths = {}
for idx, (page_id, src_path) in enumerate(page_paths.items(), 1):
    gray = cv2.imread(str(src_path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise RuntimeError(f"cannot read {src_path}")
    restored = restore_gray(gray)
    out = REST_DIR/f"{page_id}.png"
    cv2.imwrite(str(out), restored, [cv2.IMWRITE_PNG_COMPRESSION, 3])
    restored_paths[page_id] = out
    print(f"{idx:02d}/20", page_id)

del model
torch.cuda.empty_cache()

In [ ]:
# 7) Prepare pinned Oemer ONNX checkpoints and a conservative multi-class adapter.
# No training. No SVM classifiers are required in this first GPU pass.

import urllib.request
import oemer
from oemer import MODULE_PATH
from oemer.ete import generate_pred

np.int = int
np.float = float

CHECKPOINTS = {
    "unet_big/model.onnx": "https://github.com/BreezeWhite/oemer/releases/download/checkpoints/1st_model.onnx",
    "seg_net/model.onnx": "https://github.com/BreezeWhite/oemer/releases/download/checkpoints/2nd_model.onnx",
}
checkpoint_hashes = {}
for rel, url in CHECKPOINTS.items():
    dest = Path(MODULE_PATH)/"checkpoints"/rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        print("downloading", url)
        urllib.request.urlretrieve(url, dest)
    checkpoint_hashes[rel] = sha256_file(dest)

print("oemer checkpoints:", checkpoint_hashes)

def component_boxes(mask, min_area=4):
    m = (mask > 0).astype(np.uint8)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    boxes = []
    for i in range(1, n):
        x,y,w,h,area = stats[i]
        if area >= min_area and w > 0 and h > 0:
            boxes.append((int(x),int(y),int(x+w),int(y+h),int(area)))
    return boxes

def estimate_unit(staff_mask):
    occ = (staff_mask > 0).mean(axis=1)
    rows = np.flatnonzero(occ >= 0.20)
    if len(rows) < 5:
        return 10.0
    groups=[]
    for r in rows.tolist():
        if not groups or r > groups[-1][-1] + 1: groups.append([r])
        else: groups[-1].append(r)
    centers=np.array([np.mean(g) for g in groups], dtype=float)
    if len(centers) < 2: return 10.0
    diffs=np.diff(centers)
    diffs=diffs[(diffs>=3)&(diffs<=40)]
    return float(np.median(diffs)) if len(diffs) else 10.0

def iou(a,b):
    ax1,ay1,ax2,ay2=a; bx1,by1,bx2,by2=b
    ix1,iy1=max(ax1,bx1),max(ay1,by1)
    ix2,iy2=min(ax2,bx2),min(ay2,by2)
    iw,ih=max(0,ix2-ix1),max(0,iy2-iy1)
    inter=iw*ih
    if inter<=0: return 0.0
    return inter/((ax2-ax1)*(ay2-ay1)+(bx2-bx1)*(by2-by1)-inter)

def greedy_recall(src_boxes, cand_boxes, th=0.20):
    pairs=[]
    for si,s in enumerate(src_boxes):
        for ci,c in enumerate(cand_boxes):
            ov=iou(s,c)
            if ov>=th: pairs.append((ov,si,ci))
    pairs.sort(reverse=True)
    ss=set(); cc=set(); matched=0
    for ov,si,ci in pairs:
        if si in ss or ci in cc: continue
        ss.add(si); cc.add(ci); matched += 1
    return matched / max(1,len(src_boxes)), matched, max(0,len(cand_boxes)-matched)

def detect_semantic_boxes(img_path: Path):
    staff, symbols, stems_rests, notehead, clefs_keys = generate_pred(str(img_path), use_tf=False)
    unit = estimate_unit(staff)
    out = {k: [] for k in ["staff_line","tab_line","notehead","stem","beam_or_flag","rest","accidental","clef","barline"]}
    for x1,y1,x2,y2,area in component_boxes(notehead, min_area=max(3,int(unit*unit*0.08))):
        w,h=x2-x1,y2-y1
        if 0.35*unit <= w <= 2.2*unit and 0.35*unit <= h <= 2.2*unit:
            out["notehead"].append((x1,y1,x2,y2))
    sr_boxes = component_boxes(stems_rests, min_area=max(3,int(unit*0.5)))
    for x1,y1,x2,y2,area in sr_boxes:
        w,h=x2-x1,y2-y1
        if h >= 3.6*unit and w <= 1.5*unit:
            out["barline"].append((x1,y1,x2,y2))
        elif h >= 1.8*unit and w <= 1.2*unit:
            out["stem"].append((x1,y1,x2,y2))
        elif 0.45*unit <= w <= 3.0*unit and 0.45*unit <= h <= 3.6*unit:
            out["rest"].append((x1,y1,x2,y2))
    ck_boxes = component_boxes(clefs_keys, min_area=max(3,int(unit*unit*0.10)))
    for x1,y1,x2,y2,area in ck_boxes:
        w,h=x2-x1,y2-y1
        if h >= 2.4*unit and w >= 0.8*unit:
            out["clef"].append((x1,y1,x2,y2))
        elif 0.45*unit <= h <= 3.0*unit and 0.25*unit <= w <= 2.2*unit:
            out["accidental"].append((x1,y1,x2,y2))
    residue = (symbols > 0).astype(np.uint8)
    residue[(notehead>0)|(stems_rests>0)|(clefs_keys>0)|(staff>0)] = 0
    for x1,y1,x2,y2,area in component_boxes(residue, min_area=max(3,int(unit*unit*0.10))):
        w,h=x2-x1,y2-y1
        if (w >= 1.4*unit and h <= 1.8*unit) or (h >= 0.8*unit and w <= 1.8*unit):
            out["beam_or_flag"].append((x1,y1,x2,y2))
    gray = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    from st_score_restore.stage11_v2c_semantic_preservation import conservative_line_system_detector
    for d in conservative_line_system_detector(gray):
        out[d.class_id].append(tuple(map(float,d.bbox)))
    return out

In [ ]:
# 8) GPU detector benchmark: source detectability + source/restored semantic matching
teacher_pages = {p["pageId"]: p for p in teacher["pages"]}
source_dets = {}
candidate_dets = {}
source_evaluable_pairs = set()
semantic_matched_pairs = set()
pair_metrics = []

for idx, page_id in enumerate(page_paths, 1):
    print(f"[{idx:02d}/20] detector", page_id)
    sdet = detect_semantic_boxes(page_paths[page_id])
    cdet = detect_semantic_boxes(restored_paths[page_id])
    source_dets[page_id] = sdet
    candidate_dets[page_id] = cdet
    for class_id, rec in teacher_pages[page_id]["classes"].items():
        if rec["state"] != "present":
            continue
        sb = sdet.get(class_id, [])
        cb = cdet.get(class_id, [])
        if sb:
            source_evaluable_pairs.add((page_id,class_id))
            recall, matched, cand_only = greedy_recall(sb, cb, th=0.20)
            if recall >= 0.80:
                semantic_matched_pairs.add((page_id,class_id))
            pair_metrics.append({
                "pageId": page_id,
                "classId": class_id,
                "sourceCount": len(sb),
                "candidateCount": len(cb),
                "matchedCount": matched,
                "sourceRecall": recall,
                "candidateOnlyCount": cand_only,
            })

source_cov = benchmark_coverage_from_present_pairs(teacher, source_evaluable_pairs)
semantic_cov = benchmark_coverage_from_present_pairs(teacher, semantic_matched_pairs)
print("source detector coverage:", source_cov["confidentlyEvaluatedExpectedPresentClassPageCount"], "/", source_cov["eligibleExpectedPresentClassPageCount"], source_cov["applicableClassDetectorCoverage"])
print("semantic matched coverage:", semantic_cov["confidentlyEvaluatedExpectedPresentClassPageCount"], "/", semantic_cov["eligibleExpectedPresentClassPageCount"], semantic_cov["applicableClassDetectorCoverage"])

In [ ]:
# 9) Save machine-readable result to Drive. This is exploratory GPU evidence only.
import onnxruntime as ort

RESULT_DIR = DRIVE_ROOT/'V2D_RESULTS'
RESULT_DIR.mkdir(exist_ok=True)

result = {
    "schemaVersion": "stage11.v2d.detector-benchmark-result.v1",
    "contractId": "stage11.v2c.semantic-preservation.nonheldout.v1",
    "generatedOn": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "candidateId": "oemer-onnx-segmentation-dbe2a933",
    "boundary": {
        "developmentOnly": True,
        "teacherGroundTruthFrozen": True,
        "detectorOutputUsedAsGroundTruth": False,
        "trainingPerformed": False,
        "fineTuningPerformed": False,
        "heldOutAccessed": False,
        "productionPromotionAuthorized": False,
        "stage12EntryAuthorized": False,
    },
    "runtime": {
        "purpose": "exploratory_detector_benchmark",
        "device": "GPU",
        "torchVersion": torch.__version__,
        "cudaAvailable": torch.cuda.is_available(),
        "cudaDevice": torch.cuda.get_device_name(0),
        "onnxruntimeVersion": ort.__version__,
        "onnxProviders": ort.get_available_providers(),
        "restoreExecutionMode": "GPU exploratory; final canonical CPU rerun required",
        "finalCanonicalCpuRerunRequired": True,
    },
    "inputs": {
        "teacherPresentClassPageCount": 171,
        "sourceFiles": {k: {"path": str(found[k]), "sha256": TARGETS[k]["sha256"], "byteSize": TARGETS[k]["size"]} for k in found},
        "oemerUpstreamCommit": "dbe2a933d630d0f74805d717960eb259473f5978",
        "oemerCheckpointSha256": checkpoint_hashes,
    },
    "coverage": {
        "eligibleExpectedPresentClassPageCount": source_cov["eligibleExpectedPresentClassPageCount"],
        "confidentlyEvaluatedExpectedPresentClassPageCount": source_cov["confidentlyEvaluatedExpectedPresentClassPageCount"],
        "applicableClassDetectorCoverage": source_cov["applicableClassDetectorCoverage"],
        "classCoverage": source_cov["classCoverage"],
    },
    "semanticExploration": {
        "matchedAtRecallAtLeast0_80ClassPageCount": semantic_cov["confidentlyEvaluatedExpectedPresentClassPageCount"],
        "matchedAtRecallAtLeast0_80Coverage": semantic_cov["applicableClassDetectorCoverage"],
        "pairMetrics": pair_metrics,
    },
    "claimBoundary": {
        "semanticPreservationEstablished": False,
        "productionReady": False,
        "canonicalCpuEvidenceRequiredBeforeAnyUpgrade": True,
    },
}

from st_score_restore.stage11_v2d_detector_benchmark import validate_detector_benchmark_result
print(validate_detector_benchmark_result(result))

out = RESULT_DIR/'v2d_colab_gpu_detector_benchmark_result.json'
out.write_text(json.dumps(result, indent=2), encoding='utf-8')
print("SAVED:", out)

## Bittiğinde

Son hücrede `SAVED: .../V2D_RESULTS/v2d_colab_gpu_detector_benchmark_result.json` görmelisin.

Bu JSON'u **buraya yüklemen yeterli**. Sonrasında:
1. detector kapsamını sınıf sınıf analiz edeceğim,
2. zayıf sınıfları ikinci adayla veya güvenli rule-detector ile tamamlayacağım,
3. seçilen detector kombinasyonunu canonical CPU üzerinde yeniden doğrulayacağım,
4. objektif gate geçerse PR #211'i Ready/Merge aşamasına taşıyacağız.

**Not:** Bu notebook hiçbir model eğitmez.